In [1]:
import pandas as pd
import re
import string
import nltk

In [2]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

In [3]:
df = pd.read_csv("dataset.csv")
print(df.head())

                      text     label
0      I love this product  positive
1    This is a great movie  positive
2      I hate this product  negative
3      Very bad experience  negative
4  Congratulations you won      spam


In [4]:
def clean_text(text):
    text = text.lower() 
    text = re.sub(r'\d+', '', text)  
    text = text.translate(str.maketrans('', '', string.punctuation)) 
    text = text.strip()  
    return text



In [5]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    words = nltk.word_tokenize(text)
    new_words = []

    for word in words:
        if word not in stop_words:
            word = lemmatizer.lemmatize(word)
            new_words.append(word)

    return " ".join(new_words)

df["cleaned_text"] = df["text"].apply(clean_text)
df["processed_text"] = df["cleaned_text"].apply(preprocess)
print("\nProcessed Text:\n")
print(df[["text", "processed_text"]].head())


Processed Text:

                      text  processed_text
0      I love this product    love product
1    This is a great movie     great movie
2      I hate this product    hate product
3      Very bad experience  bad experience
4  Congratulations you won  congratulation


In [6]:
le = LabelEncoder()
df["encoded_label"] = le.fit_transform(df["label"])
print("\nEncoded Labels:\n")
print(df[["label", "encoded_label"]].drop_duplicates())



Encoded Labels:

      label  encoded_label
0  positive              2
2  negative              1
4      spam              3
6       ham              0


In [7]:
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(df["processed_text"])
tfidf_df = pd.DataFrame(X.toarray(), columns=tfidf.get_feature_names_out())
print("\nTF-IDF Features:\n")
print(tfidf_df.head())



TF-IDF Features:

        bad  call  congratulation  experience     great     hate  hello  let  \
0  0.000000   0.0             0.0    0.000000  0.000000  0.00000    0.0  0.0   
1  0.000000   0.0             0.0    0.000000  0.707107  0.00000    0.0  0.0   
2  0.000000   0.0             0.0    0.000000  0.000000  0.76643    0.0  0.0   
3  0.707107   0.0             0.0    0.707107  0.000000  0.00000    0.0  0.0   
4  0.000000   0.0             1.0    0.000000  0.000000  0.00000    0.0  0.0   

      love  meet     movie  prize   product  tomorrow  
0  0.76643   0.0  0.000000    0.0  0.642328       0.0  
1  0.00000   0.0  0.707107    0.0  0.000000       0.0  
2  0.00000   0.0  0.000000    0.0  0.642328       0.0  
3  0.00000   0.0  0.000000    0.0  0.000000       0.0  
4  0.00000   0.0  0.000000    0.0  0.000000       0.0  


In [8]:
df.to_csv("processed_dataset.csv", index=False)
tfidf_df.to_csv("tfidf_output.csv", index=False)
print("\nAll outputs saved successfully!")


All outputs saved successfully!
